<a href="https://colab.research.google.com/github/leenafathy/Task_1/blob/main/Copy_of_qwen_system_prompt_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen-vl-utils pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded successfully


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from PIL import Image

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
cad_code = """
translate([0.03, -0.03, -0.36])
rotate([230.00, 180.00, 180.00])
cube([1.02, 0.89, 0.73], center=true);
// BLOCK 2
translate([0.03, 0.43, 0.14])
rotate([0.00, 90.00, 180.00])
cube([1.40, 0.26, 1.02], center=true);
// BLOCK 3
translate([-0.55, 0.12, -0.23])
rotate([90.00, 180.00, 180.00])
cube([0.33, 0.69, 0.88], center=true);
// BLOCK 4
translate([0.55, 0.12, -0.23])
rotate([0.00, 0.00, 180.00])
cube([0.33, 0.88, 0.69], center=true);
"""

In [ ]:
!pip -q install datasets sentence-transformers faiss-cpu

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Garygedegege/CADReview")
print(dataset)

In [ ]:
train_data = dataset["train"]
print(train_data.column_names)
print(train_data[0])

In [ ]:
records = []

for row in train_data:
    records.append({
        "id": row["id"],
        "error_code": row["error_code"],
        "correct_code": row["correct_code"]
    })

print("Number of records:", len(records))
print(records[0].keys())

In [ ]:
def make_document(rec):
    return f"""Example ID: {rec['id']}

Erroneous CAD code:
{rec['error_code']}

Correct CAD code:
{rec['correct_code']}
"""

documents = [make_document(rec) for rec in records]

print(documents[0][:1000])

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents, show_progress_bar=True, convert_to_numpy=True)
print(doc_embeddings.shape)

In [ ]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings.astype("float32"))

print("FAISS index size:", index.ntotal)

In [ ]:
def retrieve_similar_examples(query_error_code, top_k=3):
    query_text = f"Erroneous CAD code:\n{query_error_code}"
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(records[idx])

    return results

In [ ]:
query_error_code = """
// BLOCK 1
translate([0.03, -0.03, -0.36])
rotate([230.00, 180.00, 180.00])
cube([1.02, 0.89, 0.73], center=true);

// BLOCK 2
translate([0.03, 0.43, 0.14])
rotate([0.00, 90.00, 180.00])
cube([1.40, 0.26, 1.02], center=true);

// BLOCK 3
translate([-0.55, 0.12, -0.23])
rotate([90.00, 180.00, 180.00])
cube([0.33, 0.69, 0.88], center=true);

// BLOCK 4
translate([0.55, 0.12, -0.23])
rotate([0.00, 0.00, 180.00])
cube([0.33, 0.88, 0.69], center=true);
"""

retrieved = retrieve_similar_examples(query_error_code, top_k=3)

for i, rec in enumerate(retrieved, 1):
    print(f"\n===== Retrieved Example {i} =====")
    print("ID:", rec["id"])
    print("ERROR CODE:\n", rec["error_code"][:800])
    print("\nCORRECT CODE:\n", rec["correct_code"][:800])

In [ ]:
def format_retrieved_examples(retrieved_examples):
    context = ""
    for i, rec in enumerate(retrieved_examples, 1):
        context += f"""
Example {i}
Erroneous CAD code:
{rec['error_code']}

Correct CAD code:
{rec['correct_code']}
--------------------
"""
    return context

retrieved_context = format_retrieved_examples(retrieved)
print(retrieved_context[:2000])

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Garygedegege/CADReview")
train_data = dataset["train"]

# make a split: 80% for RAG memory, 20% for testing
split_data = train_data.train_test_split(test_size=0.2, seed=42)

rag_data = split_data["train"]   # used for retrieval memory
test_data = split_data["test"]   # kept unseen for evaluation

print("RAG memory size:", len(rag_data))
print("Test size:", len(test_data))

In [ ]:
rag_records = []

for row in rag_data:
    rag_records.append({
        "id": row["id"],
        "error_code": row["error_code"],
        "correct_code": row["correct_code"]
    })

print("RAG records:", len(rag_records))
print(rag_records[0].keys())

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents, show_progress_bar=True, convert_to_numpy=True)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings.astype("float32"))

print("FAISS index size:", index.ntotal)

In [ ]:
def retrieve_similar_examples(query_error_code, top_k=3):
    query_text = f"Erroneous CAD code:\n{query_error_code}"
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(rag_records[idx])

    return results

In [ ]:
test_sample = test_data[0]

query_error_code = test_sample["error_code"]
query_image = test_sample["image"]

retrieved = retrieve_similar_examples(query_error_code, top_k=3)

print("Test sample ID:", test_sample["id"])
for i, rec in enumerate(retrieved, 1):
    print(f"Retrieved {i}:", rec["id"])

In [ ]:
def format_retrieved_examples(retrieved_examples):
    context = ""
    for i, rec in enumerate(retrieved_examples, 1):
        context += f"""
Example {i}
Erroneous CAD code:
{rec['error_code']}

Correct CAD code:
{rec['correct_code']}

--------------------
"""
    return context

retrieved_context = format_retrieved_examples(retrieved)
print(retrieved_context[:2000])

In [ ]:
system_prompt = """
You are a CAD debugging assistant.

You will be given:
1. retrieved solved examples, where each example contains an erroneous CAD code and its corrected CAD code
2. a new CAD reference image
3. a new erroneous CAD code

Your job is to use the retrieved examples as references.

For each retrieved example:
- compare the erroneous CAD code with the corrected CAD code
- identify which block changed
- identify what changed in that block
- infer the likely error type

Possible error types:
- No error
- Missing block
- Redundant block
- Size error
- Position error
- Rotation error
- Primitive error
- Logic error
- Constant error

Then analyze the new CAD image and new erroneous CAD code using the same reasoning pattern.

Rules:
- Choose only one most suspicious block in the new case
- Prefer geometric explanations such as rotation, position, size, or primitive before claiming missing block
- Be specific
- Do not say multiple blocks are wrong

Return exactly in this format:

Retrieved example analysis:
- Example 1: changed block ..., changed attribute ..., likely error type ...
- Example 2: changed block ..., changed attribute ..., likely error type ...
- Example 3: changed block ..., changed attribute ..., likely error type ...

Final answer:
Suspicious block: BLOCK X
Error type: ...
Reason: ...
Suggestion: ...
"""

In [ ]:
retrieved = retrieve_similar_examples(query_error_code, top_k=3)

In [ ]:
retrieved_context = format_retrieved_examples(retrieved)

In [ ]:
user_prompt = f"""
Here are retrieved solved CAD debugging examples:
{retrieved_context}

Now analyze the new case.

New erroneous CAD code:
{query_error_code}

Compare the new erroneous CAD code with the reference image and use the retrieved examples to infer the most likely error.
"""